# REPSOL Model Training (70/15/15)

**MobileNetV3-02 — fixed data pipeline + anti-overfitting package**

Same approach as EfficientNet-04. MobileNetV3-01 (like all runs before it) trained on a
broken view of the data: the stored `.norm.pt` files are full MATLAB figure renders
(3×1714×3156) and `dataset.py` cropped to the first 400 pixel columns — the model saw
only the leftmost ~13% of each spectrogram, mostly white margin and axis-label text.

This run uses `Data/Spectrograms_224`: the plot interior cropped from the figure,
resized to 224×224 with antialiasing, re-z-scored (~600 KB/file). Full time axis,
no text, ~10× faster epochs.

| Setting | MobileNetV3-01 | MobileNetV3-02 | Why |
|---------|----------------|----------------|-----|
| Data | first 400/3156 cols of figure render | full spectrogram, 224×224 | See all the signal |
| Augmentation | none | SpecAugment (2 freq + 2 time masks) | Cheap synthetic data |
| Backbone | all trainable | features 0–12 frozen (~22% of params) | MobileNet is back-loaded; freeze generic early filters |
| Dropout | 0.2 (default) | 0.35 | Stronger head regularisation |
| Weight decay | 1e-4 | 1e-2 | Standard AdamW fine-tuning value |
| Checkpoint metric | weighted val loss | **val macro-F1** | Weighted loss dominated by a few rare-class samples |
| LR schedule | OneCycleLR | ReduceLROnPlateau on macro-F1 | Compatible with early stopping |
| Batch size | 8 | 16 | Small inputs allow it; steadier BatchNorm |
| Class weights | balanced + ×1.5 cls 1 & 7 | unchanged | |
| Label smoothing | 0.1 | unchanged | |

## 0. Config

In [ ]:
from pathlib import Path
import sys
import torch

# ===== Hyperparameters =====
BATCH_SIZE      = 16
EPOCHS          = 30
LEARNING_RATE   = 5e-4
WEIGHT_DECAY    = 1e-2
PATIENCE        = 6      # early stopping on val macro-F1
LABEL_SMOOTHING = 0.1
DROPOUT         = 0.35
FREEZE_UP_TO    = 13     # freeze backbone.features[0:13], train 13-16 + classifier
MODEL_NAME      = "mobilenetv3_02"

# ===== Paths =====
PROJECT_ROOT    = Path(r"D:\\Work\\Internships\\INMAR\\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms_224"   # <-- fixed dataset
OUTPUT_DIR      = PROJECT_ROOT / "Models_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


def next_run_path(model_name, suffix, ext, output_dir):
    prefix = f"{model_name}{suffix}"
    existing = [0]
    for p in output_dir.iterdir():
        if not p.is_file() or p.suffix != ext:
            continue
        stem = p.stem
        if stem.startswith(prefix + "_"):
            tail = stem[len(prefix) + 1:]
            if tail.isdigit():
                existing.append(int(tail))
    return output_dir / f"{prefix}_{max(existing)+1:02d}{ext}"


CHECKPOINT_PATH = next_run_path(MODEL_NAME, "_best", ".pth", OUTPUT_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE         :", DEVICE)
print(f"LR={LEARNING_RATE}  WD={WEIGHT_DECAY}  EPOCHS={EPOCHS}  PATIENCE={PATIENCE}")
print(f"LABEL_SMOOTHING={LABEL_SMOOTHING}  DROPOUT={DROPOUT}  FREEZE_UP_TO={FREEZE_UP_TO}")

## 1. Verify Data

In [ ]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.norm.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("Tensor files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .norm.pt files found — run src/preprocess/downsize_spectrograms.py first."
assert counts["val"]   > 0, "No val .norm.pt files found."
assert counts["test"]  > 0, "No test .norm.pt files found."

sample = next((SPECTROGRAM_DIR / "train").rglob("*.norm.pt"))
t = torch.load(sample)
print(f"Sample shape: {tuple(t.shape)}  mean={t.mean():.3f}  std={t.std():.3f}")
assert t.shape[-2:] == (224, 224), f"Expected 224x224 tensors, got {tuple(t.shape)}"

## 2. Training

In [ ]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchaudio.transforms as TA
from sklearn.metrics import f1_score
from tqdm import tqdm

import src.MobileNet.model as model_module
import src.dataloaders as dl_module

model_module = importlib.reload(model_module)
dl_module    = importlib.reload(dl_module)
MobileNetV3Spectrogram = model_module.MobileNetV3Spectrogram
compute_class_weights  = model_module.compute_class_weights
get_dataloaders        = dl_module.get_dataloaders


# ── SpecAugment: random frequency + time masks, TRAIN ONLY ──
class SpecAugment(nn.Module):
    def __init__(self, freq_mask=28, time_mask=28, n_freq=2, n_time=2):
        super().__init__()
        self.freq = nn.ModuleList([TA.FrequencyMasking(freq_mask) for _ in range(n_freq)])
        self.time = nn.ModuleList([TA.TimeMasking(time_mask) for _ in range(n_time)])
    def forward(self, x):
        for m in self.freq:
            x = m(x)
        for m in self.time:
            x = m(x)
        return x


# ── DataLoaders (224x224 tensors, no pad/crop, train-time augmentation) ──
train_loader, val_loader, test_loader = get_dataloaders(
    SPECTROGRAM_DIR, batch_size=BATCH_SIZE,
    num_workers=0, pin_memory=False, persistent_workers=False,
    cache_in_memory=True,
    train_transform=SpecAugment(),
    target_width=None,
)
NUM_CLASSES = len(train_loader.dataset.classes)
CLASS_NAMES = train_loader.dataset.classes
print("Classes:", NUM_CLASSES, "  Train batches:", len(train_loader))


# ── Model: freeze early/mid feature blocks, train the heavy tail + head ──
model = MobileNetV3Spectrogram(num_classes=NUM_CLASSES, freeze_backbone=False).to(DEVICE)
for block in model.backbone.features[:FREEZE_UP_TO]:
    for p in block.parameters():
        p.requires_grad = False
model.backbone.classifier[2].p = DROPOUT   # Dropout sits at index 2; default 0.2

def set_frozen_bn_eval(m):
    """Keep BatchNorm running stats of frozen blocks fixed during training."""
    m.backbone.features[:FREEZE_UP_TO].eval()

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {n_trainable/1e6:.2f}M trainable / {n_total/1e6:.2f}M total "
      f"({100*n_trainable/n_total:.0f}%)")


# ── Class weights: sklearn balanced + ×1.5 boost for the two worst classes ──
BOOST_CLASSES = {1: 1.5, 7: 1.5}

train_labels  = [label for (_, label) in train_loader.dataset.samples]
base_weights  = compute_class_weights(train_labels, num_classes=NUM_CLASSES).numpy()
final_weights = base_weights.copy()
for cls_idx, multiplier in BOOST_CLASSES.items():
    final_weights[cls_idx] *= multiplier
class_weights = torch.tensor(final_weights, dtype=torch.float).to(DEVICE)

print("\nClass weights:")
for i, (name, bw, fw) in enumerate(zip(CLASS_NAMES, base_weights, final_weights)):
    boost = f" x{BOOST_CLASSES[i]}" if i in BOOST_CLASSES else ""
    print(f"  [{i}] {name[:45]:45s}  balanced={bw:.3f}  final={fw:.3f}{boost}")


# ── Loss / optimiser / scheduler ──
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-6)


# ── Checkpoint + history setup ──
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
HISTORY_PATH = CHECKPOINT_PATH.with_name(f"{CHECKPOINT_PATH.stem}_training_history.csv")
for p in (CHECKPOINT_PATH, HISTORY_PATH):
    if p.exists(): p.unlink()
with open(HISTORY_PATH, "w") as fh:
    fh.write("epoch,train_loss,val_loss,train_acc,val_acc,val_macro_f1,lr\n")
torch.save(model.state_dict(), CHECKPOINT_PATH)


# ── Training loop — checkpoint & early-stop on VAL MACRO-F1 (higher = better) ──
best_val_f1 = -1.0
no_improve = 0

for epoch in range(1, EPOCHS + 1):

    # --- train ---
    model.train()
    set_frozen_bn_eval(model)
    t_loss, t_correct, t_total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} Train", ncols=100, unit="batch")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        t_loss    += loss.item()
        t_correct += (out.argmax(1) == y).sum().item()
        t_total   += y.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    train_loss = t_loss / len(train_loader)
    train_acc  = 100.0 * t_correct / t_total

    # --- validate ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    all_preds, all_targets = [], []
    pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} Val  ", ncols=100, unit="batch")
    with torch.no_grad():
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out  = model(x)
            loss = criterion(out, y)
            v_loss    += loss.item()
            preds      = out.argmax(1)
            v_correct += (preds == y).sum().item()
            v_total   += y.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(y.cpu().tolist())
    val_loss = v_loss / len(val_loader)
    val_acc  = 100.0 * v_correct / v_total
    val_f1   = f1_score(all_targets, all_preds, average="macro", zero_division=0)

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"| Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} "
        f"| Train Acc: {train_acc:.2f} | Val Acc: {val_acc:.2f} "
        f"| Val macro-F1: {val_f1:.4f} | LR: {current_lr:.2e}",
        flush=True,
    )

    with open(HISTORY_PATH, "a") as fh:
        fh.write(f"{epoch},{train_loss:.6f},{val_loss:.6f},{train_acc:.4f},{val_acc:.4f},{val_f1:.6f},{current_lr:.6f}\n")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  \u2713 Saved improved checkpoint \u2192 {CHECKPOINT_PATH.name}", flush=True)
    else:
        no_improve += 1
        print(f"  No improvement {no_improve}/{PATIENCE}", flush=True)
        if no_improve >= PATIENCE:
            print("  Early stopping.", flush=True)
            break

print("\nTraining finished.")
print(f"Best val macro-F1: {best_val_f1:.4f}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## 3. Evaluation

In [ ]:
import importlib
import src.evaluate as eval_module

eval_module = importlib.reload(eval_module)
evaluate_model = eval_module.evaluate_model

# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

val_metrics  = evaluate_model(model, val_loader,  DEVICE)
test_metrics = evaluate_model(model, test_loader, DEVICE)

print("Validation:")
print({k: round(val_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})
print("\nTest:")
print({k: round(test_metrics[k], 4) for k in ["accuracy", "precision", "recall", "f1"]})

In [ ]:
print("Test Classification Report:\n")
print(test_metrics["report"])
print("Confusion Matrix:")
print(test_metrics["confusion_matrix"])

## 4. Learning Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(HISTORY_PATH)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(df["epoch"], df["train_acc"], label="train")
axes[0].plot(df["epoch"], df["val_acc"],   label="val", linestyle="--")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(df["epoch"], df["train_loss"], label="train")
axes[1].plot(df["epoch"], df["val_loss"],   label="val", linestyle="--")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(df["epoch"], df["val_macro_f1"], color="tab:green")
axes[2].set_title("Val macro-F1 (checkpoint metric)"); axes[2].set_xlabel("Epoch")

axes[3].plot(df["epoch"], df["lr"])
axes[3].set_title("Learning Rate (ReduceLROnPlateau)"); axes[3].set_xlabel("Epoch")
axes[3].set_yscale("log")

fig.suptitle("MobileNetV3-02 Learning Curves", fontsize=13)
fig.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "evaluation" / "learning_curves_mobilenetv3_02.png",
            dpi=150, bbox_inches="tight")
plt.show()